# 3.3 · 异常值检测 / Outlier Detection

> **课程定位 / Where this fits**
> 第 3 课，**Part 3 · EDA 与数据预处理**。
> Lesson 3, **Part 3 · EDA & Preprocessing**.
>
> 缺失之外，另一类常见脏数据是**异常值(outlier)**——偏离大多数样本的极端点。它可能是**录入错误**（该删），也可能是**真实的罕见事件**（该保留，甚至正是要找的目标，如欺诈）。所以异常值处理的第一原则是：**先搞清楚它是什么，再决定怎么办，不要盲目删除。**
> Besides missing data, the other common mess is the **outlier** — an extreme point far from the bulk. It can be a **data-entry error** (delete it) or a **genuine rare event** (keep it — sometimes it's exactly the target, like fraud). So rule #1: **understand what it is before acting; never blindly delete.**
>
> 💼 **实战/面试视角**：面试常问"怎么检测异常 / IQR 和 Z-score 的区别 / 多变量异常怎么办"。
> 💼 **Practical/interview angle:** "how do you detect outliers / IQR vs Z-score / multivariate outliers?"

> 💡 **面试相关 / Interview-relevant**
> - "IQR 法 vs Z-score 法的区别"（出镜率 ★★★★★）
> - "Z-score 的自我掩盖问题 / 稳健 Z-score(MAD)"（★★★★）
> - "单变量正常但组合异常怎么抓（马氏距离）"（★★★★）
> - "Isolation Forest / LOF 原理与适用"（★★★★）
> - "异常值要不要删 / winsorize vs log 变换"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 区分**单变量**与**多变量**异常。
   Distinguish univariate vs multivariate outliers.
2. 掌握 **IQR / Z-score / 稳健 Z-score(MAD)** 并理解 Z-score 的自我掩盖。
   Master IQR / Z-score / robust Z-score (MAD) and understand Z-score's self-masking.
3. 用**马氏距离**抓"每维正常但组合异常"的点。
   Use Mahalanobis distance for "normal per-axis but anomalous together" points.
4. 用 **Isolation Forest / LOF** 做模型化异常检测。
   Use Isolation Forest / LOF for model-based detection.
5. 理性**处理**异常：截断(winsorize) / 变换(log) / 标记，而非一删了之。
   Handle outliers sensibly: winsorize / log / flag — not just delete.

## 目录 / TOC
1. [先建直觉 + 单变量 vs 多变量 ⭐](#1)
2. [🏠 数据：California Housing](#2)
3. [Z-score 与自我掩盖 ⭐](#3)
4. [IQR 法 ⭐](#4)
5. [马氏距离：多变量异常 ⭐](#5)
6. [Isolation Forest / LOF ⭐](#6)
7. [怎么处理：winsorize vs log ⭐](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉 + 单变量 vs 多变量 ⭐ / Intuition & Uni- vs Multivariate

"异常"分两种，必须分清：
There are two kinds of outlier, and you must tell them apart:
- **单变量异常**：在**某一列**上取值极端（如年龄 = 999）。逐列看就能发现。
  **Univariate:** extreme on **one column** (e.g. age = 999). Found by looking column by column.
- **多变量异常**：**每一列单独看都正常**，但**组合起来违反了变量间的关系**（如身高 1.5m + 体重 120kg，各自不极端但组合反常）。逐列方法**完全抓不到**。
  **Multivariate:** **normal on each column** but the **combination violates the relationship between variables** (e.g. 1.5m tall + 120kg — neither extreme alone, but odd together). Per-column methods **miss these entirely**.

下面构造一个二维强相关数据，注入一个"每维都正常、却违反相关结构"的点，直观看清多变量异常。
Below we build 2-D strongly-correlated data and inject a point that's "normal on each axis but breaks the correlation" to make multivariate outliers concrete.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 造二维强正相关数据 / two strongly positively-correlated variables
cov = [[1, 0.85], [0.85, 1]]                      # 协方差矩阵: 0.85 = 强正相关
normal_pts = rng.multivariate_normal([0, 0], cov, 500)
mv_outlier = np.array([[2.5, -2.5]])              # x,y 各自不极端, 但一正一负违反了正相关

pts = np.vstack([normal_pts, mv_outlier])
plt.figure(figsize=(5.5, 5.5))
plt.scatter(normal_pts[:,0], normal_pts[:,1], s=12, alpha=0.5, label="normal 正常")
plt.scatter(*mv_outlier.T, s=200, c="red", marker="*", label="多变量异常 MV outlier", zorder=5)
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.legend(); plt.title("红星: x、y 各自都在正常范围内\n但 (2.5,-2.5) 违反正相关 → 多变量异常")
plt.tight_layout(); plt.show()
print(f"红星 x={mv_outlier[0,0]} (|z|={abs(mv_outlier[0,0]):.1f}, 不极端), y={mv_outlier[0,1]} (|z|={abs(mv_outlier[0,1]):.1f}, 不极端)")
print("逐列 z-score 都抓不到! 需要马氏距离(第5节) / per-column z-score misses it → need Mahalanobis")


<a id="2"></a>
## 2. 数据：California Housing / The California Housing Dataset

**California Housing**（加州街区房价数据，sklearn 内置）：每行一个街区，特征有收入中位数、房龄、平均房间数、人口等。真实数据里就藏着错误——某些街区的"平均房间数"高达 100+，明显是数据问题，正好用来练手。
**California Housing** (block-level housing data, built into sklearn): one row per block, with median income, house age, average rooms, population, etc. The real data already contains errors — some blocks show 100+ "average rooms", clearly bad data, perfect for practice.


In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df = data.frame[["MedInc","HouseAge","AveRooms","Population"]].copy()
print(f"shape: {df.shape}")
print(df.describe().round(2))
print(f"\nAveRooms 最大值 max = {df.AveRooms.max():.0f} (平均一户 100+ 房间?? 明显数据错误 data error)")


<a id="3"></a>
## 3. Z-score 与自我掩盖 ⭐ / Z-score & Self-Masking

**Z-score 法**：算每个点偏离均值多少个标准差 $z=\frac{x-\bar x}{s}$，$|z|>3$ 视为异常。简单，但有个致命缺陷——**自我掩盖(masking)**：均值和标准差**本身会被异常值带偏**，几个大异常值会互相抬高 std，结果谁都不超过 3，集体逃脱检测。
**Z-score:** measure how many standard deviations a point is from the mean, $z=\frac{x-\bar x}{s}$, flagging $|z|>3$. Simple, but has a fatal flaw — **masking**: the mean and std are **themselves dragged by the outliers**; several big outliers inflate the std so none exceeds 3, and they all escape detection.

**稳健 Z-score** 用**中位数**和 **MAD**（中位数绝对偏差）替代均值和 std——中位数/MAD 对异常值不敏感，不会被带偏。
The **robust Z-score** replaces mean and std with the **median** and **MAD** (median absolute deviation), which are insensitive to outliers and won't be dragged.


In [ ]:
x = df["MedInc"].values

# 普通 z-score: 用均值和标准差 / classic z-score with mean & std
z = (x - x.mean()) / x.std()
# 稳健 z-score: 用中位数和 MAD(中位数绝对偏差); 1.4826 让 MAD 在正态下等价于 std
med = np.median(x)
mad = np.median(np.abs(x - med))
z_robust = (x - med) / (1.4826 * mad)
print(f"普通 z-score   (|z|>3): 抓到 {(np.abs(z)>3).sum()} 个异常")
print(f"稳健 z-score   (|z|>3): 抓到 {(np.abs(z_robust)>3).sum()} 个异常")

# 演示自我掩盖: 人为加 5 个超大值, 看普通 z-score 怎么被骗 / demonstrate masking
print("\n=== 自我掩盖 masking ===")
x_dirty = np.append(x, [50, 60, 70, 80, 90])              # 注入 5 个极端值
z_dirty = (x_dirty - x_dirty.mean()) / x_dirty.std()     # 这 5 个值互相抬高了 std
print(f"加污染后这 5 个点的 |z| = {np.abs(z_dirty[-5:]).round(1)}")
print(f"其中 |z|>3 的只有 {(np.abs(z_dirty[-5:])>3).sum()} 个 — 它们互相抬高 std, 掩盖了彼此!")
# 稳健版用中位数/MAD, 不被污染带偏, 全部抓到
z_rob_dirty = (x_dirty - np.median(x_dirty)) / (1.4826*np.median(np.abs(x_dirty-np.median(x_dirty))))
print(f"稳健版全部抓到: |z_robust|>3 有 {(np.abs(z_rob_dirty[-5:])>3).sum()}/5 ✓")


<a id="4"></a>
## 4. IQR 法 ⭐ / The IQR Rule

**IQR 法**（箱线图背后的规则）：用四分位距 $IQR=Q_3-Q_1$，把 $[Q_1-1.5\,IQR,\ Q_3+1.5\,IQR]$ 之外的点视为异常。它基于**分位数**，天生**对异常值稳健**（不像 Z-score 依赖均值/std），是单变量异常检测的默认首选，也是箱线图"须"的定义。
**The IQR rule** (behind the boxplot): with the interquartile range $IQR=Q_3-Q_1$, flag points outside $[Q_1-1.5\,IQR,\ Q_3+1.5\,IQR]$. Being **quantile-based**, it's **naturally robust** (unlike Z-score's mean/std), the default for univariate detection, and the definition of a boxplot's whiskers.


In [ ]:
def iqr_outliers(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])     # 第一、第三四分位数
    iqr = q3 - q1                            # 四分位距
    lo, hi = q1 - k*iqr, q3 + k*iqr          # 上下界(k=1.5 是常用经验值, k=3 更宽松)
    return (x < lo) | (x > hi), (lo, hi)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, df.columns):
    mask, (lo, hi) = iqr_outliers(df[col].values)
    sns.boxplot(y=df[col], ax=ax)           # 箱线图: 须之外的点就是 IQR 异常
    ax.set_title(f"{col}\n{mask.sum()} outliers ({mask.mean():.1%})")
plt.tight_layout(); plt.show()
print("箱线图须(whisker)的端点就是 Q1-1.5IQR / Q3+1.5IQR; 之外的点即 IQR 法异常")


<a id="5"></a>
## 5. 马氏距离：多变量异常 ⭐ / Mahalanobis Distance

回到第 1 节那个"每维正常、组合异常"的点。普通欧氏距离按各方向**等权**衡量距离，抓不到它。**马氏距离**先用协方差矩阵**把数据的相关结构"拉直"**，再量距离——于是违反相关结构的点会得到很大的马氏距离。
Back to the "normal-per-axis, anomalous-together" point. Plain Euclidean distance weighs all directions equally and misses it. **Mahalanobis distance** first uses the covariance matrix to **"whiten" the correlation structure**, then measures distance — so a point that violates the correlation gets a large Mahalanobis distance.

$$D_M(\mathbf{x}) = \sqrt{(\mathbf{x}-\boldsymbol\mu)^\top \boldsymbol\Sigma^{-1} (\mathbf{x}-\boldsymbol\mu)}$$

阈值用卡方分布（马氏距离平方在正态下服从 $\chi^2$）。
The threshold comes from the chi-square distribution (squared Mahalanobis is $\chi^2$ under normality).


In [ ]:
from scipy.stats import chi2

mu = normal_pts.mean(axis=0)
Sigma = np.cov(normal_pts.T)                 # 协方差矩阵(编码了相关结构)
Sigma_inv = np.linalg.inv(Sigma)            # 它的逆, 用来"拉直"相关

def mahalanobis(X, mu, Sigma_inv):
    diff = X - mu
    # 逐行算 (x-μ)ᵀ Σ⁻¹ (x-μ) 再开方; 这里用逐元素乘+求和的向量化写法
    return np.sqrt(np.sum(diff @ Sigma_inv * diff, axis=1))

dm_outlier = mahalanobis(mv_outlier, mu, Sigma_inv)[0]
thresh = np.sqrt(chi2.ppf(0.975, df=2))     # 自由度=维数=2 的 97.5% 分位作阈值
print(f"多变量异常点的马氏距离 Mahalanobis = {dm_outlier:.2f}")
print(f"它的普通欧氏距离 Euclidean = {np.sqrt((mv_outlier[0]**2).sum()):.2f} (并不远!)")
print(f"chi2 阈值 threshold (97.5%) = {thresh:.2f}")
print(f"→ 马氏距离 {dm_outlier:.1f} >> 阈值 {thresh:.1f}, 成功抓到! 而 z-score 每维都<3 完全漏掉")


<a id="6"></a>
## 6. Isolation Forest / LOF ⭐ / Model-based Detection

高维数据用规则法力不从心，转向**模型化**异常检测（这也是 6.16 的内容，这里从预处理角度先用）：
For high-dim data, rule-based methods struggle; turn to **model-based** detection (also covered in 6.16; here from a preprocessing angle):
- **Isolation Forest**：随机切分空间，**异常点因为孤立，更少几刀就被隔离**（路径短）。快、适合高维大数据，是首选。
  **Isolation Forest:** randomly partitions space; **anomalies, being isolated, get separated in fewer cuts** (short path). Fast, great for high-dim big data — the go-to.
- **LOF（局部离群因子）**：比较一个点与其邻居的**局部密度**，显著更稀疏的就是异常。擅长**多密度**数据，但**对尺度敏感，要先标准化**。
  **LOF (Local Outlier Factor):** compares a point's **local density** to its neighbors'; much sparser = outlier. Good for **varying-density** data, but **scale-sensitive — standardize first**.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

X = df.values
# Isolation Forest: contamination=预期异常比例(这里设 2%) / expected outlier fraction
iso = IsolationForest(contamination=0.02, random_state=0)
labels = iso.fit_predict(X)          # 返回 -1=异常, 1=正常
scores = iso.score_samples(X)        # 异常分数(越小越异常)
print(f"Isolation Forest 标记 {(labels==-1).sum()} 个异常 ({(labels==-1).mean():.1%})")
print("最异常的 5 个街区 most anomalous blocks:")
print(df.assign(score=scores).sort_values("score").head(5).round(1))

# LOF: 对尺度敏感, 先标准化再做 / scale-sensitive, standardize first
Xs = StandardScaler().fit_transform(X)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
lof_labels = lof.fit_predict(Xs)
overlap = ((labels==-1) & (lof_labels==-1)).sum()
print(f"\nLOF 标记 {(lof_labels==-1).sum()} 个; IF 与 LOF 都标记的 {overlap} 个(方法不同, 部分重叠正常)")
print("IF 适合高维大数据; LOF 适合多密度簇; 实践常两者都跑, 取交集(更稳)或并集(更全)")


<a id="7"></a>
## 7. 怎么处理：winsorize vs log ⭐ / Handling: Winsorize vs Log

**检测到异常 ≠ 必须删除**。盲删会丢信息、还可能删掉真实的重要事件。更常用的处理：
**Detecting an outlier ≠ must delete it.** Blind deletion loses information and may discard genuinely important events. More common handling:
- **截断(winsorize)**：把超过某分位（如 98%）的值**压到该分位**，保留样本只削峰。
  **Winsorize:** cap values beyond a quantile (e.g. 98%) **to that quantile** — keep the row, just clip the peak.
- **log 变换**：对右偏数据取 log，**自然地压缩长尾**，往往是最优雅的方案（不删任何数据）。
  **Log transform:** for right-skewed data, take the log to **compress the tail naturally** — often the most elegant fix (deletes nothing).
- **标记**：加一个"是否异常"的指示列，让模型自己决定（同 3.2 缺失指示符思路）。
  **Flag:** add an "is-outlier" indicator column and let the model decide (like the missingness indicator, 3.2).
- **只删确认的错误**：如那些 100+ 房间的街区。
  **Delete only confirmed errors:** like the 100+ rooms blocks.

> ⚠️ 和填补一样，winsorize 的分位、变换的参数**都只能在训练集上算**，再套到测试集（防泄漏）。
> ⚠️ Like imputation, winsorize quantiles and transform parameters must be computed **on train only**, then applied to test (prevent leakage).


In [ ]:
from scipy.stats.mstats import winsorize
x = df["MedInc"].values

wins = winsorize(x, limits=[0, 0.02])    # 把最高的 2% 压到 98 分位 / cap top 2%
logged = np.log1p(x)                      # log(1+x): 压缩右尾, 能处理 0

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].hist(x, bins=50); axes[0].set_title(f"原始 original (skew={st.skew(x):.2f})")
axes[1].hist(wins, bins=50); axes[1].set_title(f"winsorize 2% (skew={st.skew(wins):.2f})")
axes[2].hist(logged, bins=50); axes[2].set_title(f"log1p (skew={st.skew(logged):.2f})")
plt.tight_layout(); plt.show()
print("log 变换常是处理右偏'异常'最优雅的方案 — 不删数据, 自然压缩长尾(偏度大幅下降)")
print("Log transform is often the most elegant fix for right-skewed 'outliers' — deletes nothing.")


<a id="8"></a>
## 8. 小结 / Summary

```
两类异常: 单变量(某列极端, 逐列可查) vs 多变量(每列正常但组合反常, 逐列漏)
Z-score: |z|>3; 但有自我掩盖(均值/std 被异常带偏) → 用稳健 Z-score(中位数+MAD)
IQR 法: Q1-1.5IQR ~ Q3+1.5IQR 之外; 分位数基, 天生稳健; = 箱线图须
马氏距离: 用 Σ⁻¹ 拉直相关结构后量距离 → 抓多变量异常; 阈值用 χ²
Isolation Forest(高维大数据首选) / LOF(多密度, 需标准化)
处理: winsorize(截断) / log(变换) / 标记 / 只删确认错误; 别盲删! 参数只在 train 算
```

### 💡 面试速查 / Interview cheat-sheet
1. **IQR 稳健、Z-score 不稳健**（均值/std 会被异常带偏 → 用 MAD 版）。
   IQR is robust; Z-score isn't (mean/std get dragged → use the MAD version).
2. **多变量异常用马氏距离**（逐列方法漏掉"组合异常"）。
   Multivariate outliers need Mahalanobis (per-column methods miss combined anomalies).
3. **Isolation Forest** 高维大数据首选；**LOF** 多密度但要标准化。
   Isolation Forest for high-dim big data; LOF for varying density (standardize first).
4. **检测 ≠ 删除**：winsorize / log / 标记，盲删丢信息且可能删掉关键事件。
   Detection ≠ deletion: winsorize / log / flag; blind deletion loses info.
5. **处理参数只在训练集算**（防泄漏）。
   Compute handling parameters on train only (prevent leakage).

### 下一节 / Next
**3.4 特征缩放**——很多模型(KNN/SVM/线性/神经网络)对量纲敏感。标准化、归一化、稳健缩放的区别与选择。
**3.4 Feature Scaling** — many models (KNN/SVM/linear/NN) are scale-sensitive. Standardization, normalization, robust scaling: differences and choices.
